In [0]:
%sql
CREATE VOLUME IF NOT EXISTS genai.spark_lab

In [0]:
import requests

# Define the current catalog
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# Define the base path using the current catalog
volume_base = f"/Volumes/{catalog_name}/genai/spark_lab"

# List of files to download
files = ["2019.csv", "2020.csv", "2021.csv"]

# Download each file
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DatabricksGenAIEngineer/refs/heads/main/Databricks_Fundamentals/{file}"
    response = requests.get(url)
    response.raise_for_status()

    # Write to Unity Catalog volume
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

In [0]:
df = spark.read.load(f'/Volumes/{catalog_name}/genai/spark_lab/*.csv', format='csv')
display(df.limit(100))

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
orderSchema = StructType([
    StructField("SalesOrderNumber", StringType()),
    StructField("SalesOrderLineNumber", IntegerType()),
    StructField("OrderDate", DateType()),
    StructField("CustomerName", StringType()),
    StructField("Email", StringType()),
    StructField("Item", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("UnitPrice", FloatType()),
    StructField("Tax", FloatType())
])
df = spark.read.load(f'/Volumes/{catalog_name}/genai/spark_lab/*.csv', format='csv', schema=orderSchema)
display(df.limit(100))

In [0]:
df.createOrReplaceTempView("salesorders")
spark_df = spark.sql("SELECT * FROM salesorders")
display(spark_df)

In [0]:
sqlQuery = "SELECT CAST(YEAR(OrderDate) AS CHAR(4)) AS OrderYear, \
               SUM((UnitPrice * Quantity) + Tax) AS GrossRevenue \
        FROM salesorders \
        GROUP BY CAST(YEAR(OrderDate) AS CHAR(4)) \
        ORDER BY OrderYear"
df_spark= spark.sql(sqlQuery)
df_spark.show()

In [0]:
sqlQuery1 = "SELECT  date_format(OrderDate, 'yyyy') AS OrderYear,\
SUM(UnitPrice * Quantity + Tax) AS GrossRevenue FROM salesorders \
GROUP BY date_format(OrderDate, 'yyyy') \
ORDER BY OrderYear; "
df_spark1 = spark.sql(sqlQuery1)
df_spark1.show()